In [50]:
#Importing dependencies 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

In [51]:
df = pd.read_csv("/users/imbahndu/Desktop/Columbia DBM/Typing dataset/modeling.csv")

In [52]:
df

,Hand,HoldTime,Direction,LatencyTime,FlightTime,Parkinsons,Gender_Male
0,0,78.1,0,289.1,195.3,True,False
1,0,130.9,0,220.7,175.8,True,False
2,1,85.9,3,257.8,171.9,False,True
3,0,78.1,0,226.6,132.8,True,True
4,0,117.2,2,171.9,93.8,False,True
...,...,...,...,...,...,...,...
742400,1,109.4,3,312.5,218.8,True,False
742401,0,93.8,2,218.8,125.0,True,True
742402,1,78.1,3,265.6,187.5,False,True
742403,1,109.4,1,296.9,203.1,True,False


In [53]:
df.isnull().sum()

Hand           0
HoldTime       0
Direction      0
LatencyTime    0
FlightTime     0
Parkinsons     0
Gender_Male    0
dtype: int64

In [54]:
df["Parkinsons"].value_counts()


Parkinsons
False    427655
True     314750
Name: count, dtype: int64

In [55]:
#Dataset was already preprocessed and balanced, so jump right into modeling 

#defining labels
from sklearn.model_selection import train_test_split

y = df["Parkinsons"]
X = df.drop(columns = ["Parkinsons", "Gender_Male", "Hand", "Direction"])

#Balancing dataset
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state = 42)

X_train_raw, y_train_raw = smote.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(X_train_raw, y_train_raw, test_size = 0.2, random_state = 42)

/users/imbahndu/.local/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [56]:
print(X_train)

print(y_train)

        HoldTime  LatencyTime  FlightTime
182566     121.1        308.6       168.0
481540      82.0        175.8       109.4
848985      85.9        250.0       164.1
812542     117.2        203.1       105.5
303709      78.1        312.5       242.2
...          ...          ...         ...
259178      93.8        269.5       175.8
365838     132.8        289.1       195.3
131932      85.9        265.6       187.5
671155      85.9        265.6       171.9
121958      97.7        304.7       183.6

[684248 rows x 3 columns]
182566     True
481540     True
848985     True
812542     True
303709     True
          ...  
259178     True
365838     True
131932     True
671155    False
121958     True
Name: Parkinsons, Length: 684248, dtype: bool


In [57]:
#training model
model_xgb = XGBClassifier()

model_xgb.fit(X_train, y_train)

pred = model_xgb.predict(X_test)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

acc_score = accuracy_score(y_test, pred)

acc_score

0.6251592989676258

In [58]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

       False       0.63      0.62      0.62     85610
        True       0.62      0.63      0.63     85452

    accuracy                           0.63    171062
   macro avg       0.63      0.63      0.63    171062
weighted avg       0.63      0.63      0.63    171062



In [59]:
#Class is so freaking imbalanced--> will adjust this demain using sklearn.utils resample 

#This is the best i could get so keeping this and try sklearn.utils demain and also maybe try SMOTE and RandomOverSampler on the entire dataset. Do this for both tyoing and this dataset

In [60]:
feat = model_xgb.feature_importances_

In [61]:
pd.DataFrame({"Column": X_train.columns, "Feature" :feat})

,Column,Feature
0,HoldTime,0.561432
1,LatencyTime,0.215396
2,FlightTime,0.223173


In [62]:
#Training a Random Forest Classifer
from sklearn.ensemble import RandomForestClassifier

random = RandomForestClassifier()

random.fit(X_train, y_train)

pred = random.predict(X_test)

print(accuracy_score(y_test, pred))

0.6339631244811823


In [ ]:
#Keep random forest